# LeJEPA Pretraining Playground

Interactive testing of pretraining configs. Uses the actual training code from `experiments/pretrain/`.

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath('..')
DATA_DIR = os.path.join(REPO_ROOT, 'data')
PRETRAIN = os.path.join(REPO_ROOT, 'experiments', 'pretrain')

sys.path.insert(0, PRETRAIN)
sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))
sys.path.insert(0, REPO_ROOT)

import torch, time
import matplotlib.pyplot as plt

from configs import Config, DATASET_INFO
from data import get_dataloaders, InMemoryGPUDataset
from models import LeJEPAEncoder, LinearProbe
from losses import build_regularizer, build_sigreg
from scheduler import make_scheduler
from trainer import setup_seed
from train_loops import (
    train_epoch_standard_inmem,
    train_epoch_standard_loader,
    train_epoch_pooled_loader,
    make_nograd_loader,
    evaluate,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Data dir: {DATA_DIR}')
print(f'Available datasets: {list(DATASET_INFO.keys())}')

## Setup & Train

In [ ]:
cfg = Config(
    dataset='cifar10',
    data_dir=DATA_DIR,
    encoder_scale='resnet18',  # 'resnet18' / 'resnet34' / 'tiny' / 'small' / ...
    regularizer='w1',          # 'sigreg', 'w1', 'w2'
    accumulate=False,          # True = pooled 2-step
    batch_size=256,
    epochs=5,
    use_compile=False,
    num_workers=4,
    eval_interval=1,
    log_interval=100,
    seed=42,
)

print(f'Dataset: {cfg.dataset} ({cfg.num_classes} classes)')
print(f'Backbone: {cfg.backbone_name}')
print(f'Multi-crop: V_g={cfg.num_global_views}@{cfg.global_crop_size} + V_l={cfg.num_local_views}@{cfg.local_crop_size}')
print(f'Regularizer: {cfg.regularizer}, pooled={cfg.accumulate}')
print(f'BS={cfg.batch_size}, epochs={cfg.epochs}, lr={cfg.lr}, λ={cfg.lambd}')

In [ ]:
def run_training(cfg):
    """Run training for given config, return history dict."""
    setup_seed(cfg.seed)
    train_source, val_source, gpu_aug = get_dataloaders(cfg, device)
    in_memory = isinstance(train_source, InMemoryGPUDataset)

    encoder = LeJEPAEncoder(cfg).to(device)
    probe = LinearProbe(encoder.hidden_dim, cfg.num_classes).to(device)
    print(f'Encoder params: {sum(p.numel() for p in encoder.parameters()):,}')

    enc_opt = torch.optim.AdamW(encoder.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    probe_opt = torch.optim.AdamW(probe.parameters(), lr=cfg.probe_lr, weight_decay=cfg.probe_wd)

    if in_memory:
        steps_per_epoch = len(train_source) // cfg.batch_size
    else:
        steps_per_epoch = len(train_source)
    total_steps = cfg.epochs * steps_per_epoch
    warmup_steps = cfg.warmup_epochs * steps_per_epoch

    enc_sched = make_scheduler(enc_opt, warmup_steps, total_steps, cfg.lr, cfg.eta_min)
    probe_sched = make_scheduler(probe_opt, warmup_steps, total_steps, cfg.probe_lr, 1e-5)

    reg_fn = None
    sigreg_mod = None
    nograd_loader = None
    nograd_iter_state = [None]

    if not cfg.accumulate:
        reg_fn = build_regularizer(cfg, device)
    else:
        if in_memory:
            raise NotImplementedError(
                'Pooled (accumulate=True) requires DataLoader path; '
                'use a non-cached dataset.')
        if cfg.regularizer == 'sigreg':
            sigreg_mod = build_sigreg(cfg, device)
        nograd_bs = (cfg.accum_steps - 1) * cfg.batch_size
        nograd_loader = make_nograd_loader(
            train_source.dataset, nograd_bs, cfg.num_workers)
        nograd_iter_state = [iter(nograd_loader)]

    sample_gen = torch.Generator(device=device).manual_seed(cfg.seed) if in_memory else None
    history = {'epoch': [], 'train_loss': [], 'val_acc': []}
    global_step = 0

    for epoch in range(cfg.epochs):
        t0 = time.time()
        if cfg.accumulate:
            avg_loss, global_step = train_epoch_pooled_loader(
                epoch, encoder, probe, train_source,
                gpu_aug, nograd_loader, nograd_iter_state,
                enc_opt, probe_opt, enc_sched, probe_sched, cfg,
                global_step, sigreg_mod)
        elif in_memory:
            avg_loss, global_step = train_epoch_standard_inmem(
                epoch, encoder, probe, reg_fn, train_source,
                gpu_aug, enc_opt, probe_opt, enc_sched, probe_sched,
                cfg, global_step, sample_gen)
        else:
            avg_loss, global_step = train_epoch_standard_loader(
                epoch, encoder, probe, reg_fn, train_source,
                gpu_aug, enc_opt, probe_opt, enc_sched, probe_sched,
                cfg, global_step)

        val_acc = evaluate(encoder, probe, val_source, cfg)
        elapsed = time.time() - t0

        history['epoch'].append(epoch)
        history['train_loss'].append(avg_loss)
        history['val_acc'].append(val_acc)

        print(f'Epoch {epoch} | loss={avg_loss:.4f} val_acc={val_acc:.4f} | {elapsed:.1f}s')

    return history

In [ ]:
history = run_training(cfg)

## Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(history['epoch'], history['train_loss'])
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Train Loss')

axes[1].plot(history['epoch'], history['val_acc'])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Val Accuracy')

fig.suptitle(f'{cfg.dataset} | {cfg.regularizer} {"pooled" if cfg.accumulate else "mini"} | BS={cfg.batch_size}')
fig.tight_layout()
plt.show()

## Compare: Mini-batch vs Pooled

In [ ]:
DATASET = 'cifar10'
BS = 64
EPOCHS = 10

results = {}
for reg in ['w1', 'sigreg']:
    for pooled in [False, True]:
        label = f'{reg} {"pooled" if pooled else "mini"}'
        print(f'\n=== {label} ===')
        c = Config(
            dataset=DATASET, data_dir=DATA_DIR,
            encoder_scale='resnet18', regularizer=reg,
            accumulate=pooled, batch_size=BS, epochs=EPOCHS,
            use_compile=False, num_workers=4, eval_interval=1,
            log_interval=9999, seed=42,
        )
        results[label] = run_training(c)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for label, h in results.items():
    ls = '-' if 'pooled' in label else '--'
    color = 'tab:blue' if 'w1' in label else 'tab:orange'
    ax.plot(h['epoch'], h['val_acc'], label=label, linestyle=ls, color=color, linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Val Accuracy')
ax.set_title(f'{DATASET} BS={BS}')
ax.legend()
plt.show()